# 00 · VectorBT 快速入门

依次运行全部单元格。默认用 2025 上半年快速验证完整生产规则；确认环境后可把日期改为十年区间。

In [ ]:
from pathlib import Path
import sys

start = Path.cwd().resolve()
ROOT = next((candidate for candidate in (start, *start.parents) if (candidate / 'vbt').exists()), start)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'项目根目录: {ROOT}')

## 1. 环境检查

In [ ]:
import vectorbt, pandas, numpy, pyarrow, duckdb
print({'vectorbt': vectorbt.__version__, 'pandas': pandas.__version__, 'numpy': numpy.__version__})

## 2. 加载配置与数据

In [ ]:
from vbt.config import load_backtest_config, load_strategy_config
from vbt.adapters import VBTDataLoader

config = load_backtest_config({'start_date': '2025-01-01', 'end_date': '2025-06-30'})
params = load_strategy_config()
data = VBTDataLoader(start_date=config['start_date'], end_date=config['end_date']).load_aligned()
data.metadata

## 3. 运行完整规则回测

In [ ]:
from vbt.engine import VBTEngine, PerformanceCalculator, ReportGenerator
from vbt.strategies import DividendLowVolStrategy

engine = VBTEngine(data=data, strategy=DividendLowVolStrategy(params), initial_capital=config['initial_capital'],
    commission=config['commission'], min_commission=config['min_commission'],
    stamp_duty_before=config['stamp_duty_before_2023_08_28'], stamp_duty_after=config['stamp_duty_after_2023_08_28'],
    slippage=config['slippage'], backtest_config=config)
results = engine.run()
perf = PerformanceCalculator(results)
perf.compute_metrics()

## 4. 图表与持仓

In [ ]:
import pandas as pd
display(results.nav.to_frame('组合资产').plot(figsize=(12, 4), title='VectorBT 组合净值').get_figure())
display(ReportGenerator(results, perf, params).current_holdings())

## 5. 一键归档

In [ ]:
paths = ReportGenerator(results, perf, params).archive(config['output_dir'])
paths